In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

class Length(layers.Layer):
    def call(self, inputs, **kwargs):
        return tf.sqrt(tf.reduce_sum(tf.square(inputs), -1) + 1e-7)

class PrimaryCaps(layers.Layer):
    def __init__(self, dim_capsule, n_channels, kernel_size, strides, padding, **kwargs):
        super(PrimaryCaps, self).__init__(**kwargs)
        self.conv = layers.Conv2D(filters=dim_capsule * n_channels,
                                  kernel_size=kernel_size,
                                  strides=strides,
                                  padding=padding,
                                  activation='relu')
        self.dim_capsule = dim_capsule
        self.n_channels = n_channels

    def call(self, inputs, **kwargs):
        output = self.conv(inputs)
        output_shape = tf.shape(output)
        output = tf.reshape(output, (-1, output_shape[1] * output_shape[2] * self.n_channels, self.dim_capsule))
        return self.squash(output)


    def squash(self, s, axis=-1):
        s_norm = tf.reduce_sum(tf.square(s), axis, keepdims=True)
        scale = s_norm / (1 + s_norm) / tf.sqrt(s_norm + 1e-7)
        return scale * s

class DigitCaps(layers.Layer):
    def __init__(self, num_capsule, dim_capsule, routings=3, **kwargs):
        super(DigitCaps, self).__init__(**kwargs)
        self.num_capsule = num_capsule
        self.dim_capsule = dim_capsule
        self.routings = routings
        self.W = None

    def build(self, input_shape):
        self.input_num_capsule = input_shape[1]
        self.input_dim_capsule = input_shape[2]
        self.W = self.add_weight(
            shape=[1, self.input_num_capsule, self.num_capsule, self.input_dim_capsule, self.dim_capsule],
            initializer='glorot_uniform',
            trainable=True
        )

    def call(self, inputs):
        batch_size = tf.shape(inputs)[0]
        inputs_expand = tf.expand_dims(tf.expand_dims(inputs, 2), -1)
        inputs_tiled = tf.tile(inputs_expand, [1, 1, self.num_capsule, 1, 1])
        W_tiled = tf.tile(self.W, [batch_size, 1, 1, 1, 1])

        u_hat = tf.matmul(W_tiled, inputs_tiled, transpose_a=True)
        u_hat = tf.squeeze(u_hat, axis=-1)

        b = tf.zeros_like(u_hat[:, :, :, 0])

        for i in range(self.routings):
            c = tf.nn.softmax(b, axis=2)
            s = tf.reduce_sum(c[..., tf.newaxis] * u_hat, axis=1)
            v = self.squash(s)
            if i < self.routings - 1:
                b += tf.reduce_sum(u_hat * tf.expand_dims(v, 1), axis=-1)


        return v

    def squash(self, s, axis=-1):
        s_norm_sq = tf.reduce_sum(tf.square(s), axis, keepdims=True)
        scale = s_norm_sq / (1 + s_norm_sq)
        return scale * s / (tf.sqrt(s_norm_sq) + tf.keras.backend.epsilon())

def CapsNet(input_shape, num_classes, routings):
    inputs = keras.Input(shape=input_shape)
    x = layers.Conv2D(256, 9, activation='relu')(inputs)
    x = PrimaryCaps(dim_capsule=8, n_channels=32, kernel_size=9, strides=2, padding='valid')(x)
    x = DigitCaps(num_capsule=num_classes, dim_capsule=16, routings=routings)(x)
    output = Length()(x)
    return keras.Model(inputs=inputs, outputs=output)

(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

x_train = tf.expand_dims(x_train, axis=-1)
x_test = tf.expand_dims(x_test, axis=-1)

y_train = tf.one_hot(y_train, depth=10)
y_test = tf.one_hot(y_test, depth=10)

model = CapsNet(input_shape=[28, 28, 1], num_classes=10, routings=3)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model.summary()

model.fit(x_train, y_train, batch_size=64, epochs=5, validation_data=(x_test, y_test))

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 20, 20, 256)    │        20,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ primary_caps (PrimaryCaps)      │ (None, 1152, 8)        │     5,308,672 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ digit_caps (DigitCaps)          │ (None, 10, 16)         │     1,474,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ length (Length)                 │ (None, 10)             │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,804,224 (25.96 MB)

 Trainable params: 6,804,224 (25.96 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 3684s 4s/step - accuracy: 0.9369 - loss: 0.2404 - val_accuracy: 0.9861 - val_loss: 0.0738
Epoch 2/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.9881 - loss: 0.0652